<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/codeAlongs/WRIT20833_YouTube_Comment_Collector_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRIT 20833 — YouTube Comment Collector

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this notebook by going to **File > Save a copy in Drive**

This notebook collects public comments from a YouTube video **you** choose and turns them into a
clean table you can save as a CSV file. You'll bring that file to HW3 (sentiment) and HW4 (topics),
and maybe to your capstone.

**How this notebook works.** You change exactly **one** cell, the ✏️ settings cell in Step 2. After
that, you run every cell from top to bottom. The code cells come in two kinds:

- **📖 Read this one.** It uses things you already know: variables, lists, loops, functions,
  dictionaries, pandas. Read it and make sure you could explain it.
- **🔧 Plumbing.** It talks to YouTube's servers or handles errors. Run it; you don't need to read
  every line yet. It's borrowed infrastructure, like the libraries you install, not a concept you're
  expected to master today.

If a cell gives you an error, read the message: most of them say exactly what to fix. Errors are
part of the work here, not something to hide.

# Public Comments Are Still Human Expression

Before collecting anything, think about what you're about to collect. Each comment is a person
saying something, usually quickly, usually to the other people watching that video, never to you.

- **Public isn't the same as fair game.** Anyone can read these comments, but that doesn't erase
  your obligations to the people who wrote them. They didn't agree to be part of a class project.
- **Comments aren't public opinion.** The people who comment are a small, self-selected slice of
  the people who watched, and the people who watched are a slice of everyone who cares about the
  topic. A news channel's audience, a comedian's audience, and an activist's audience will argue
  about the same law in different ways.
- **The sort order chooses for you.** "Most relevant" is YouTube's ranking: it favors comments with
  lots of likes and replies, and the formula is secret. "Most recent" favors whoever posted last.
  Neither one is a **random** sample, and neither one is **representative**. Name the one you used
  in your writeup, because it shapes what you find.
- **Computers miss context.** Sarcasm, irony, in-jokes, quotes from the video, and cultural
  references are easy for a person to catch and hard for VADER or a topic model to catch.
- **The archive has already been edited.** Commenters delete comments, channels moderate them, and
  YouTube filters spam. What you collect is what survived, on the day you collected it.

**What this notebook does about it.** It keeps no usernames and no channel IDs. It collects
only what your analysis needs. When you write about your data, prefer **patterns and paraphrase**
over long verbatim quotes: a quoted comment can be pasted into a search engine and traced back to
its author, even without a username. Quote only when the exact words matter to your argument.

This is the course's standing question in a new place: **what does this flatten, who chose it, and
can you trust it?**

# Step 1: Get a YouTube API Key (one time only)

An **API** (application programming interface) is a door a company builds so programs can ask for
data in an orderly way, instead of scraping web pages. YouTube's door requires a **key**, a long
password-like string that identifies you. It's free; you do this once and reuse the key all semester.

**1. Create a Google Cloud project.** Go to [console.cloud.google.com](https://console.cloud.google.com),
sign in, and accept the terms if asked. Click the project menu at the top → **New Project** → name it
something like `writ20833` → **Create**. Make sure the new project is selected in that top menu.

**2. Turn on the YouTube Data API v3.** In the search bar at the top, type **YouTube Data API v3**,
open it, and click **Enable**.

**3. Create an API key.** Go to **APIs & Services → Credentials → + Create credentials → API key**.
Copy the key. (Optional but wise: click **Edit API key**, and under *API restrictions* choose
**Restrict key → YouTube Data API v3** so the key can't be used for anything else. Leave
*Application restrictions* at **None**: Colab can't pass a website or IP-address restriction.)
No credit card or billing is needed for this.

**4. Store the key in Colab Secrets.** In this notebook, click the **🔑 key icon** in the left
sidebar → **Add new secret**. Set **Name** to exactly `YOUTUBE_API_KEY`, paste your key as the
**Value**, and switch **Notebook access** on.

**Keep your key private.** Treat it like a password: don't paste it into a code cell, a discussion
post, a screenshot, or a chat. Secrets belong to *your* Google account, not to the notebook, so
sharing this notebook with someone else doesn't share your key. If a key ever leaks, delete it on
the Credentials page and make a new one.

**How much can you collect?** Each project gets 10,000 free "units" a day, and each request
costs 1 unit. One request brings back up to 100 comments, so collecting 500 comments costs about 6
units. You won't run out unless you collect replies from a very large video many times over.

# Step 2: Your Settings ✏️

This is the **only** cell you need to change. In Colab, these lines also appear as a small form to the
right of the code, so you can type in boxes instead of editing the code.

- **`VIDEO_URL`**: the link to your video. Any of these forms works: `youtube.com/watch?v=…`,
  `youtu.be/…`, `youtube.com/shorts/…`.
- **`NUMBER_OF_COMMENTS`**: how many to collect (default 500). If the video has fewer, you get what's
  there. If you include replies, they count toward this number too.
- **`INCLUDE_REPLIES`**: `False` collects only top-level comments (replies to the video). `True` also
  collects replies to those comments, which are conversations *between commenters* and read differently.
- **`SORT_ORDER`**: `"Most relevant"` (YouTube's ranking) or `"Most recent"` (newest first).

In [ ]:
# ✏️ YOUR SETTINGS: the only cell you need to edit

VIDEO_URL = ""  # @param {type:"string"}
NUMBER_OF_COMMENTS = 500  # @param {type:"integer"}
INCLUDE_REPLIES = False  # @param {type:"boolean"}
SORT_ORDER = "Most relevant"  # @param ["Most relevant", "Most recent"]

# Step 3: Set Up

Two plumbing cells: load the tools, then load your key from Colab Secrets. The key goes into a
variable called `API_KEY`, and **nothing in this notebook ever prints it**.

In [ ]:
# 🔧 PLUMBING: load the tools this notebook uses
import json
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
import requests

API_URL = "https://www.googleapis.com/youtube/v3/"

# Translate the settings-cell choices into the codes YouTube's API expects
SORT_CODES = {"Most relevant": "relevance", "Most recent": "time"}

# The columns of our finished table, in order
COLUMNS = ["comment_id", "video_id", "video_title", "comment_text", "published_at",
           "like_count", "reply_count", "is_reply", "parent_id"]

# Count requests as we go, so we know how much of the daily quota we used
api_usage = {"requests": 0}

print("✅ Tools loaded.")

In [ ]:
# 🔧 PLUMBING: read your API key from Colab Secrets (the 🔑 icon in the left sidebar)
# ("raise ... from None" keeps each error message short and to the point)
try:
    from google.colab import userdata
except ImportError:
    raise RuntimeError("This notebook reads your key from Colab Secrets, "
                       "so it needs to run in Google Colab.") from None

try:
    API_KEY = userdata.get("YOUTUBE_API_KEY")
except userdata.SecretNotFoundError:
    raise RuntimeError(
        "No secret named YOUTUBE_API_KEY. Open the 🔑 Secrets panel on the left, add a secret "
        "with exactly that name, paste your key as its value, and run this cell again.") from None
except userdata.NotebookAccessError:
    raise RuntimeError(
        "Your YOUTUBE_API_KEY secret exists, but this notebook isn't allowed to read it. In the "
        "🔑 Secrets panel, switch on 'Notebook access' next to it, then run this cell again.") from None

if API_KEY is None or API_KEY.strip() == "":
    raise RuntimeError("Your YOUTUBE_API_KEY secret is empty. "
                       "Paste your key as its value in the 🔑 Secrets panel.")

API_KEY = API_KEY.strip()
print("✅ API key loaded (not shown, on purpose).")

# Step 4: From a Link to a Video ID

YouTube identifies every video by an **ID** of 11 characters, the part after `v=` in a normal link.
The API wants the ID, not the whole link. Links come in several shapes, so this function uses
string methods you already know (`in`, `.split()`, `len()`) to find the ID in any of them.

In [ ]:
# 📖 READ THIS ONE: pull the 11-character video ID out of a YouTube link
def get_video_id(url):
    url = url.strip()

    if "youtu.be/" in url:                  # short links: youtu.be/ID
        video_id = url.split("youtu.be/")[1]
    elif "/shorts/" in url:                 # Shorts: youtube.com/shorts/ID
        video_id = url.split("/shorts/")[1]
    elif "/live/" in url:                   # livestreams: youtube.com/live/ID
        video_id = url.split("/live/")[1]
    elif "/embed/" in url:                  # embedded players: youtube.com/embed/ID
        video_id = url.split("/embed/")[1]
    elif "v=" in url:                       # normal links: youtube.com/watch?v=ID
        video_id = url.split("v=")[1]
    else:                                   # maybe it's already just the ID
        video_id = url

    # Links often carry extras after the ID ("&t=30s", "?si=abc"); cut them off
    for separator in ["&", "?", "#", "/"]:
        video_id = video_id.split(separator)[0]

    if len(video_id) != 11:
        raise ValueError(f"Couldn't find a video ID in {url!r}. Paste the full link from your "
                         "browser's address bar (or the Share button) into VIDEO_URL in Step 2.")
    return video_id


# Try it on three link shapes (made-up ID, just to test the function)
examples = ["https://www.youtube.com/watch?v=abcDEF12345&t=30s",
            "https://youtu.be/abcDEF12345?si=xyz",
            "https://www.youtube.com/shorts/abcDEF12345"]
for example in examples:
    print(get_video_id(example), "<-", example)

In [ ]:
# 🔧 PLUMBING: check the settings, then find YOUR video's ID
if VIDEO_URL.strip() == "":
    raise ValueError("VIDEO_URL is empty. Paste a YouTube link into the settings cell in Step 2, "
                     "run that cell, then run this one again.")
if type(NUMBER_OF_COMMENTS) != int or NUMBER_OF_COMMENTS < 1:
    raise ValueError("NUMBER_OF_COMMENTS should be a whole number, 1 or more (like 500).")
if SORT_ORDER not in SORT_CODES:
    raise ValueError(f'SORT_ORDER should be "Most relevant" or "Most recent", not {SORT_ORDER!r}.')

VIDEO_ID = get_video_id(VIDEO_URL)
print("Video ID:", VIDEO_ID)

# Step 5: Asking YouTube for Data

Every request to the API is a web address plus some **parameters**: which video, how many results,
what order. That makes a parameter set a **dictionary**. YouTube answers with a dictionary too, often
with lists and dictionaries nested inside it.

The function below sends one request and hands back the answer. Most of its lines handle things going
wrong (no internet, a bad key, comments turned off, the daily quota used up) and turn YouTube's
technical error codes into plain-English messages.

In [ ]:
# 🔧 PLUMBING: send one request to the YouTube Data API and return the answer as a dictionary

# YouTube names each problem with a short "reason" code; these are the ones you're likely to meet
FRIENDLY_ERRORS = {
    "commentsDisabled": "Comments are turned off for this video, so there's nothing to collect. "
                        "Try a different video.",
    "videoNotFound": "YouTube can't find that video. It may be private or deleted, "
                     "or the link may be mistyped.",
    "forbidden": "YouTube refused this request. For comments, that usually means the channel has turned "
                 "comments off or limited who can see them, so try a different video. If it happens on "
                 "every video, check your key's restrictions in Google Cloud (APIs & Services > Credentials).",
    "quotaExceeded": "Your key has used up today's free quota. It resets at midnight Pacific time. "
                     "Try again tomorrow, or collect fewer comments or turn replies off.",
    "dailyLimitExceeded": "Your key has used up today's free quota. It resets at midnight Pacific time.",
    "rateLimitExceeded": "Too many requests too quickly. Wait a minute, then run the cell again.",
    "userRateLimitExceeded": "Too many requests too quickly. Wait a minute, then run the cell again.",
    "keyInvalid": "YouTube rejected your API key. Copy it again from Google Cloud "
                  "(APIs & Services > Credentials) and update your YOUTUBE_API_KEY secret.",
    "API_KEY_INVALID": "YouTube rejected your API key. Copy it again from Google Cloud "
                       "(APIs & Services > Credentials) and update your YOUTUBE_API_KEY secret.",
    "accessNotConfigured": "The YouTube Data API v3 isn't turned on for your Google Cloud project. "
                           "Enable it (Step 1, part 2), wait a minute, and try again.",
    "SERVICE_DISABLED": "The YouTube Data API v3 isn't turned on for your Google Cloud project. "
                        "Enable it (Step 1, part 2), wait a minute, and try again.",
    "API_KEY_SERVICE_BLOCKED": "Your key isn't allowed to use the YouTube Data API. In Google Cloud, "
                               "edit the key's API restrictions to include YouTube Data API v3.",
    "API_KEY_HTTP_REFERRER_BLOCKED": "Your key is restricted to certain websites, and Colab isn't one of "
                                     "them. In Google Cloud, edit the key and set Application "
                                     "restrictions to None (keep the API restriction).",
    "API_KEY_IP_ADDRESS_BLOCKED": "Your key is restricted to certain IP addresses, and Colab's aren't "
                                  "among them. In Google Cloud, edit the key and set Application "
                                  "restrictions to None (keep the API restriction).",
}


def ask_youtube(endpoint, params):
    params = dict(params)            # a copy, so the key is never added to the caller's dictionary
    params["key"] = API_KEY
    api_usage["requests"] += 1

    try:
        response = requests.get(API_URL + endpoint, params=params, timeout=30)
    except requests.RequestException:
        # Don't show the original error: its text includes the request's web address, which holds your key
        raise RuntimeError("Couldn't reach YouTube. Check your internet connection "
                           "and run the cell again.") from None

    if response.status_code == 200:
        return response.json()

    # Something went wrong: find YouTube's reason code(s) and translate them.
    # "details" holds the more specific codes, so check those first.
    try:
        error = response.json()["error"]
    except (ValueError, KeyError):
        error = {}
    reasons = []
    for item in error.get("details", []) + error.get("errors", []):
        reasons.append(item.get("reason", ""))

    explanation = f"YouTube returned an error (HTTP {response.status_code})."
    for reason in reasons:
        if reason in FRIENDLY_ERRORS:
            explanation = FRIENDLY_ERRORS[reason]
            break
    # Always add YouTube's own words and which request failed: they help you (or your instructor) diagnose it
    raise RuntimeError(f"{explanation}\n\nYouTube's own words: {error.get('message', 'no details given')}"
                       f"\n(request: {endpoint}; reason code: {', '.join(reasons) or 'none'})")

First request: ask about the **video itself**. This confirms the video exists and is public,
and gets its title for your records.

In [ ]:
# 📖 READ THIS ONE: look up the video's title and channel
video_answer = ask_youtube("videos", {"part": "snippet,statistics", "id": VIDEO_ID})

if len(video_answer.get("items", [])) == 0:
    raise RuntimeError("YouTube found no public video with that ID. The video may be private or "
                       "deleted, or the link may be mistyped. Check VIDEO_URL in Step 2.")

video = video_answer["items"][0]                 # the answer holds a list with one video in it
VIDEO_TITLE = video["snippet"]["title"]
CHANNEL_TITLE = video["snippet"]["channelTitle"]
reported_count = video["statistics"].get("commentCount")   # missing when comments are off

print("Title:  ", VIDEO_TITLE)
print("Channel:", CHANNEL_TITLE)
print("Posted: ", video["snippet"]["publishedAt"][:10])
if reported_count is None:
    print("⚠️ YouTube shows no comment count for this video, so comments are probably turned off.")
else:
    reported_count = int(reported_count)
    print("Comments YouTube reports:", reported_count, "(including replies)")

# Step 6: Collect the Comments

YouTube hands out comments in **pages** of up to 100. Each answer includes a `nextPageToken`, a
bookmark you send back to get the next page. When an answer has no bookmark, you've reached the end.

Here's the shape of one comment *thread* in YouTube's answer, trimmed down. These are placeholders,
not a real comment:

```
{"id": "Ugx…",
 "snippet": {"totalReplyCount": 3,
             "topLevelComment": {"id": "Ugx…",
                                 "snippet": {"textOriginal": "(the comment's words)",
                                             "publishedAt": "2025-07-08T14:02:11Z",
                                             "likeCount": 12,
                                             "authorDisplayName": "(the commenter's name)", …}}},
 "replies": {"comments": [ …up to 5 replies… ]}}
```

The answer includes the commenter's name. **We deliberately don't copy it**, as the next cell shows.

In [ ]:
# 📖 READ THIS ONE: turn one comment from YouTube's answer into one row for our table
def make_row(comment, is_reply, reply_count):
    snippet = comment["snippet"]
    text = snippet.get("textOriginal") or snippet.get("textDisplay") or ""
    return {
        "comment_id": comment["id"],
        "video_id": VIDEO_ID,
        "video_title": VIDEO_TITLE,
        "comment_text": text,
        "published_at": snippet.get("publishedAt", ""),
        "like_count": snippet.get("likeCount", 0),
        "reply_count": reply_count,
        "is_reply": is_reply,
        "parent_id": snippet.get("parentId"),   # only replies have a parent; None otherwise
        # No author name, channel, or profile picture: we don't need them, so we don't keep them.
    }


# A dictionary can't hold the same key twice, so storing rows by comment_id prevents duplicates
collected = {}


def add_row(row):
    """Add a row unless we already have that comment. Returns True if it was new."""
    if row["comment_id"] in collected:
        return False
    collected[row["comment_id"]] = row
    return True

In [ ]:
# 🔧 PLUMBING: fetch ALL of one thread's replies (a thread's answer includes at most 5)
def get_all_replies(parent_id, still_needed):
    replies = []
    page_token = None
    while len(replies) < still_needed:
        params = {"part": "snippet", "parentId": parent_id,
                  "maxResults": 100, "textFormat": "plainText"}
        if page_token:
            params["pageToken"] = page_token
        answer = ask_youtube("comments", params)
        for reply in answer.get("items", []):
            replies.append(reply)
        page_token = answer.get("nextPageToken")
        if not page_token:
            break
    return replies

The main loop. Read it slowly: it's a `while` loop around a `for` loop, plus a few `if`s.

1. Ask for a page of comment threads.
2. For each thread, add the top-level comment. If you asked for replies, add those too.
3. Stop when you have enough comments, or when YouTube has no more pages.

In [ ]:
# 📖 READ THIS ONE: the collection loop
collected.clear()          # start fresh if you run this cell twice
duplicates_skipped = 0
page_token = None
page_number = 0

if INCLUDE_REPLIES:
    parts = "snippet,replies"
else:
    parts = "snippet"

while len(collected) < NUMBER_OF_COMMENTS:
    params = {"part": parts, "videoId": VIDEO_ID, "maxResults": 100,
              "order": SORT_CODES[SORT_ORDER], "textFormat": "plainText"}
    if page_token:
        params["pageToken"] = page_token
    answer = ask_youtube("commentThreads", params)
    page_number = page_number + 1
    new_on_this_page = 0

    for thread in answer.get("items", []):
        if len(collected) >= NUMBER_OF_COMMENTS:
            break
        top_comment = thread["snippet"]["topLevelComment"]
        total_replies = thread["snippet"].get("totalReplyCount", 0)

        if not add_row(make_row(top_comment, False, total_replies)):
            duplicates_skipped = duplicates_skipped + 1
            continue                       # already have this thread (and its replies)
        new_on_this_page = new_on_this_page + 1

        if INCLUDE_REPLIES and total_replies > 0:
            replies = thread.get("replies", {}).get("comments", [])
            if len(replies) < total_replies:          # the thread came back short: fetch the rest
                replies = get_all_replies(top_comment["id"], NUMBER_OF_COMMENTS - len(collected))
            for reply in replies:
                if len(collected) >= NUMBER_OF_COMMENTS:
                    break
                if not add_row(make_row(reply, True, 0)):
                    duplicates_skipped = duplicates_skipped + 1

    print(f"Page {page_number}: {len(collected)} comments so far")

    page_token = answer.get("nextPageToken")
    if not page_token:
        print("Reached the end of the comments YouTube will share for this video.")
        break
    if new_on_this_page == 0:
        print("That page held only comments we already had, so we stopped.")
        break

print()
print(f"Collected {len(collected)} comments using {api_usage['requests']} API requests.")
if len(collected) < NUMBER_OF_COMMENTS:
    print(f"(You asked for {NUMBER_OF_COMMENTS}; this is everything that was available.)")
if duplicates_skipped > 0:
    print(f"Skipped {duplicates_skipped} duplicates that YouTube sent more than once.")

# Step 7: Make the Table

`collected` is a dictionary of rows, and each row is itself a dictionary. pandas turns a list of
dictionaries straight into a table, with one row per dictionary and one column per key.

In [ ]:
# 📖 READ THIS ONE: from a dictionary of rows to a DataFrame
comments_df = pd.DataFrame(list(collected.values()), columns=COLUMNS)

if len(comments_df) == 0:
    print("⚠️ No comments came back, so there's nothing to clean or save. Try another video.")

comments_df.head()

Also keep a record of **how** you collected the data: when, how many you asked for, and how you
sorted. This record is part of your method, and it goes in your writeup. "I collected 500 'most
relevant' comments on November 13" tells a reader what your sample can and can't show. It's stored
separately from the comments, as a dictionary you can save as a small JSON file in Step 10.

In [ ]:
# 📖 READ THIS ONE: a record of how this dataset was made
collection_info = {
    "collected_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "video_id": VIDEO_ID,
    "video_url": "https://www.youtube.com/watch?v=" + VIDEO_ID,
    "video_title": VIDEO_TITLE,
    "channel_title": CHANNEL_TITLE,
    "comment_count_reported_by_youtube": reported_count,
    "requested_sample_size": NUMBER_OF_COMMENTS,
    "comments_collected": len(comments_df),
    "sort_order": SORT_ORDER,
    "replies_included": INCLUDE_REPLIES,
    "duplicates_skipped": duplicates_skipped,
    "api_requests_used": api_usage["requests"],
    "source": "YouTube Data API v3 (videos.list, commentThreads.list, comments.list)",
}
collection_info

# Step 8: Clean, Gently

In Pandas 02 you cleaned a messy table. YouTube's comments come back tidier than a scraper's, but
not perfect: some comments are only whitespace, and many have line breaks in the middle.

**What we fix:**
- Make a **new** column, `comment_clean`, where runs of spaces, tabs, and line breaks become a
  single space. `comment_text` keeps the original exactly as it was posted.
- Drop comments that are empty once the whitespace is gone.
- Check for repeated comment IDs. There shouldn't be any, but checking is cheap.

**What we deliberately *don't* fix:** capital letters, punctuation, emoji, "not," "lol," "/s."
Compare:

| original | "aggressively cleaned" |
|---|---|
| `This is NOT okay!!! 😡` | `okay` |
| `Oh great, ANOTHER law. Just what we needed /s` | `oh great another law needed` |

*(Two invented examples, not real comments.)* Lowercasing, stripping punctuation and emoji, and
dropping stopwords like "not" turned an angry comment into a mild one and sarcasm into praise.
**VADER reads capitals, "!!!", emoji, and "not"** as signals, so it needs them. HW4's topic
modeling does its own heavier cleaning *inside* that notebook, on a copy, which leaves your saved
file untouched.

In [ ]:
# 📖 READ THIS ONE: collapse whitespace into a new column; the original stays as it was
def normalize_whitespace(text):
    # .split() with no argument splits on ANY run of spaces, tabs, or line breaks;
    # " ".join() glues the pieces back together with single spaces
    return " ".join(str(text).split())


clean_df = comments_df.copy()             # never change your raw data in place
clean_df["comment_clean"] = clean_df["comment_text"].apply(normalize_whitespace)

# Put comment_clean right after comment_text, so they sit side by side
clean_df = clean_df[COLUMNS[:4] + ["comment_clean"] + COLUMNS[4:]]

before = len(clean_df)
clean_df = clean_df[clean_df["comment_clean"] != ""]                   # drop empty comments
empty_removed = before - len(clean_df)

before = len(clean_df)
clean_df = clean_df.drop_duplicates(subset="comment_id").reset_index(drop=True)
duplicate_ids_removed = before - len(clean_df)

print("Empty comments removed:      ", empty_removed)
print("Duplicate comment IDs removed:", duplicate_ids_removed)
print("Rows in the clean table:     ", len(clean_df))

collection_info["empty_comments_removed"] = empty_removed
collection_info["rows_after_cleaning"] = len(clean_df)

**Same words, different people?** Many commenters post the identical short text ("First!", "❤️",
"This."). Those are separate comments by separate people, so we **keep** them. Spam-bot repeats are
also identical, though, and those you might drop. That's a judgment call about your data, not an
automatic step, so the next cell only *shows* you the repeats.

In [ ]:
# 📖 READ THIS ONE: which exact texts appear more than once? (shown, not removed)
repeated_texts = clean_df["comment_clean"].value_counts()
repeated_texts = repeated_texts[repeated_texts > 1]
print(len(repeated_texts), "texts appear more than once")
repeated_texts.head(10)

# Step 9: A First Look

A few quick numbers, to check that what you collected matches what you meant to collect. This
isn't analysis yet; HW3 and HW4 are where that happens.

In [ ]:
# 📖 READ THIS ONE: the basic shape of your dataset
word_counts = clean_df["comment_clean"].str.split().str.len()
dates = pd.to_datetime(clean_df["published_at"])

print("Total comments:      ", len(clean_df))
print("Top-level comments:  ", (clean_df["is_reply"] == False).sum())
print("Replies:             ", (clean_df["is_reply"] == True).sum())
if len(clean_df) > 0:
    print("Average length:      ", round(word_counts.mean(), 1), "words /",
          round(clean_df["comment_clean"].str.len().mean(), 1), "characters")
    print("Shortest / longest:  ", word_counts.min(), "/", word_counts.max(), "words")
    print("Posted between:      ", dates.min().date(), "and", dates.max().date())

**Five comments at random.** `.head()` would show the first five, which are the top of
YouTube's sort order, so `.sample()` gives a fairer glimpse. (Change `random_state` to see five others.)

In [ ]:
# 📖 READ THIS ONE: five comments chosen at random
sample = clean_df.sample(min(5, len(clean_df)), random_state=1)
for text in sample["comment_clean"]:
    print("•", text[:300])
    print()

**The ten most frequent words**, with common function words ("the," "and," "is") filtered out, as
in HW2. The lowercasing and punctuation-stripping here happen **only for this count**; the table
itself isn't changed.

In [ ]:
# 📖 READ THIS ONE: the ten most frequent words (a quick look, not your final analysis)
STOPWORDS = {"the", "a", "an", "and", "or", "but", "of", "to", "in", "on", "for", "with", "at", "by",
             "from", "is", "are", "was", "were", "be", "been", "it", "its", "it's", "this", "that",
             "i", "you", "he", "she", "we", "they", "me", "my", "your", "our", "their", "his", "her",
             "them", "so", "as", "if", "not", "no", "do", "does", "did", "have", "has", "had", "just",
             "what", "who", "all", "can", "will", "would", "about", "there", "than", "then", "out",
             "up", "how", "when", "more", "i'm", "don't", "like", "get", "one", "us"}

word_counter = Counter()
for text in clean_df["comment_clean"]:
    for word in text.lower().split():
        word = word.strip(".,!?;:\"'()[]{}…“”‘’*-_")
        if word != "" and word not in STOPWORDS:
            word_counter[word] += 1

word_counter.most_common(10)

*Notice:* "not" is in that stopword list, which is fine for counting topics but would be a disaster
for sentiment. The same word can be noise for one method and signal for another.

# Step 10: Save Your Dataset

This saves `clean_df` as a CSV named after the video's ID and downloads it to your computer. Both
`comment_text` (original) and `comment_clean` are in the file. UTF-8 encoding keeps emoji and accented
letters intact.

**Colab forgets files** when your session ends, so download the CSV (or copy it to your Google
Drive) before you close the tab.

**If you open the CSV in Excel** and see clutter like `‚ÄúSo kids‚Äù` where curly quotes should be,
Excel has guessed the wrong encoding. The file is fine, and Python and Google Sheets read it correctly.
This notebook saves with `"utf-8-sig"`, which adds an invisible marker telling Excel "this is UTF-8."
If you still see clutter, import the file through Excel's **Data → From Text/CSV** and choose UTF-8.

In [ ]:
# 📖 READ THIS ONE: save the CSV, then reload it to check that nothing got lost
csv_filename = f"youtube_comments_{VIDEO_ID}.csv"
# "utf-8-sig" is UTF-8 plus an invisible marker at the start, so Excel reads emoji and curly quotes correctly
clean_df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

# keep_default_na=False stops pandas from turning a comment that says "NA" or "None" into a blank
check = pd.read_csv(csv_filename, encoding="utf-8", keep_default_na=False)
same_rows = len(check) == len(clean_df)
same_text = list(check["comment_text"].astype(str)) == list(clean_df["comment_text"].astype(str))
print("Saved", csv_filename)
print("Reloads with the same rows:", same_rows, "| same text, emoji included:", same_text)

In [ ]:
# 🔧 PLUMBING: download the CSV to your computer
from google.colab import files
files.download(csv_filename)

**Optional:** download the collection record from Step 7 as a JSON file too. Keep it beside your
CSV; it's the source of the details your methods paragraph needs.

In [ ]:
# 🔧 PLUMBING (optional): save and download the collection record
info_filename = f"youtube_comments_{VIDEO_ID}_info.json"
with open(info_filename, "w", encoding="utf-8") as f:
    json.dump(collection_info, f, indent=2, ensure_ascii=False)
files.download(info_filename)

# Where This Goes Next

**In HW3 (sentiment) and HW4 (topics),** upload your CSV to Colab (📁 Files panel → upload) and
fill in the two "Choose your data" lines in the setup cell:

```python
MY_CSV = "youtube_comments_YOURVIDEOID.csv"
MY_TEXT_COLUMN = "comment_clean"
```

The setup cell reads it with `keep_default_na=False` (so a comment that says "NA" stays text) and
prints a `DATA SOURCE:` line. Check that line names your file before you go on.

**One row = one whole comment.** The course corpus split long comments into several rows; yours
doesn't. So in HW3 each score is one person's whole comment, which makes your counts easier to
interpret.

**For your writeup, record:** the video and channel, the date you collected, how many comments you
asked for and got, the sort order, and whether replies were included. All of it is in
`collection_info`. And keep asking the question from the top of this notebook: whose voices did
this sample let in, and whose did it leave out?

# Playground

In [ ]:
# use this space to experiment!